# Text Summarizer with LSTM + Attention in PyTorch

This Kaggle notebook trains a sequence-to-sequence text summarizer on the Amazon Fine Food Reviews dataset. It uses the review `Text` column as the source text and the `Summary` column as the target summary.

Before running on Kaggle, add the **Amazon Fine Food Reviews** dataset to the notebook. The code automatically searches `/kaggle/input` for `Reviews.csv`.

In [1]:
import os
import re
import html
import math
import time
import random
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


## Configuration

The default settings are designed to run comfortably on Kaggle. Increase `MAX_ROWS`, vocab sizes, hidden size, or epochs for better results if you have more GPU time.

In [2]:
MAX_ROWS = 100_000
MAX_SOURCE_LEN = 120
MAX_TARGET_LEN = 18
MIN_SOURCE_LEN = 15
MIN_TARGET_LEN = 2

SRC_MIN_FREQ = 3
TGT_MIN_FREQ = 2
MAX_SRC_VOCAB = 30_000
MAX_TGT_VOCAB = 12_000

BATCH_SIZE = 64
EMBED_DIM = 128
ENC_HIDDEN_DIM = 256
DEC_HIDDEN_DIM = 256
DROPOUT = 0.30
EPOCHS = 5
LEARNING_RATE = 1e-3
CLIP = 1.0
TEACHER_FORCING_RATIO = 0.5

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"

PAD_IDX = 0
UNK_IDX = 1
SOS_IDX = 2
EOS_IDX = 3

## Load The Dataset

In [3]:
def find_reviews_csv():
    search_roots = ["/kaggle/input", "."]
    for root_dir in search_roots:
        if not os.path.exists(root_dir):
            continue
        for root, _, files in os.walk(root_dir):
            if "Reviews.csv" in files:
                return os.path.join(root, "Reviews.csv")
    raise FileNotFoundError("Reviews.csv was not found. Add the Amazon Fine Food Reviews dataset to this Kaggle notebook.")

csv_path = find_reviews_csv()
print("Using dataset:", csv_path)

df = pd.read_csv(csv_path, usecols=["Text", "Summary"])
df = df.dropna().drop_duplicates().reset_index(drop=True)

if MAX_ROWS is not None and len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)

print(df.shape)
df.head()

Using dataset: /kaggle/input/datasets/akhileshchittoria/textsummarizer/Reviews.csv
(100000, 2)


,Summary,Text
0,If You Like to Cook Healthy for Your Family.....,"If you want to cook healthily for your family,..."
1,Some package and origin information,"The package reads: ""Product of USA, China, Tha..."
2,the best oils for my holiday candy,I have never had such great turnout for my Chr...
3,mimic creme healthy top,This is very good. It surpassed my expectatio...
4,Unbelievably great product!!!,My wife is an amazing cook -- and she's Italia...


## Clean And Prepare Text

In [4]:
CONTRACTIONS = {
    "can't": "cannot",
    "won't": "will not",
    "n't": " not",
    "'re": " are",
    "'s": " is",
    "'d": " would",
    "'ll": " will",
    "'t": " not",
    "'ve": " have",
    "'m": " am",
}

def clean_text(text):
    text = html.unescape(str(text)).lower()
    text = re.sub(r"<.*?>", " ", text)
    for src, dst in CONTRACTIONS.items():
        text = text.replace(src, dst)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["source"] = df["Text"].map(clean_text)
df["target"] = df["Summary"].map(clean_text)

df["source_len"] = df["source"].str.split().map(len)
df["target_len"] = df["target"].str.split().map(len)

df = df[
    (df["source_len"].between(MIN_SOURCE_LEN, MAX_SOURCE_LEN))
    & (df["target_len"].between(MIN_TARGET_LEN, MAX_TARGET_LEN))
].reset_index(drop=True)

print(df.shape)
df[["source", "target", "source_len", "target_len"]].head()

(72194, 6)


,source,target,source_len,target_len
0,i have never had such great turnout for my chr...,the best oils for my holiday candy,47,7
1,this is very good it surpassed my expectations...,mimic creme healthy top,27,4
2,my wife is an amazing cook and she is italian ...,unbelievably great product,89,3
3,this coffee is ok i have yet to buy a k cup th...,very strong,66,2
4,i have purchased these treats twice for my mot...,picky dog loves them,36,4


In [5]:
train_df, valid_df = train_test_split(df, test_size=0.10, random_state=SEED)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Valid:", valid_df.shape)

Train: (64974, 6)
Valid: (7220, 6)


## Build Vocabularies

In [6]:
def build_vocab(texts, min_freq, max_size):
    counter = Counter()
    for text in texts:
        counter.update(text.split())

    specials = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]
    words = [word for word, freq in counter.most_common() if freq >= min_freq]
    words = words[: max(0, max_size - len(specials))]
    itos = specials + words
    stoi = {word: idx for idx, word in enumerate(itos)}
    return stoi, itos

src_stoi, src_itos = build_vocab(train_df["source"], SRC_MIN_FREQ, MAX_SRC_VOCAB)
tgt_stoi, tgt_itos = build_vocab(train_df["target"], TGT_MIN_FREQ, MAX_TGT_VOCAB)

print("Source vocab size:", len(src_itos))
print("Target vocab size:", len(tgt_itos))

Source vocab size: 16762
Target vocab size: 6884


## Dataset And DataLoader

In [7]:
def encode_source(text):
    tokens = text.split()[:MAX_SOURCE_LEN]
    return [src_stoi.get(token, UNK_IDX) for token in tokens]

def encode_target(text):
    tokens = text.split()[:MAX_TARGET_LEN]
    ids = [SOS_IDX]
    ids.extend(tgt_stoi.get(token, UNK_IDX) for token in tokens)
    ids.append(EOS_IDX)
    return ids

class ReviewSummaryDataset(Dataset):
    def __init__(self, frame):
        self.sources = frame["source"].tolist()
        self.targets = frame["target"].tolist()

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        return torch.tensor(encode_source(self.sources[idx]), dtype=torch.long), torch.tensor(encode_target(self.targets[idx]), dtype=torch.long)

def collate_batch(batch):
    sources, targets = zip(*batch)
    source_lengths = torch.tensor([len(src) for src in sources], dtype=torch.long)
    target_lengths = torch.tensor([len(tgt) for tgt in targets], dtype=torch.long)

    max_src_len = max(source_lengths).item()
    max_tgt_len = max(target_lengths).item()

    padded_sources = torch.full((len(batch), max_src_len), PAD_IDX, dtype=torch.long)
    padded_targets = torch.full((len(batch), max_tgt_len), PAD_IDX, dtype=torch.long)

    for i, (src, tgt) in enumerate(zip(sources, targets)):
        padded_sources[i, : len(src)] = src
        padded_targets[i, : len(tgt)] = tgt

    return padded_sources, source_lengths, padded_targets

train_loader = DataLoader(
    ReviewSummaryDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

valid_loader = DataLoader(
    ReviewSummaryDataset(valid_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

batch = next(iter(train_loader))
print(batch[0].shape, batch[1].shape, batch[2].shape)

torch.Size([64, 120]) torch.Size([64]) torch.Size([64, 13])


## LSTM Encoder-Decoder With Bahdanau Attention

In [8]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embed_dim, enc_hidden_dim, dec_hidden_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embed_dim, padding_idx=PAD_IDX)
        self.rnn = nn.LSTM(embed_dim, enc_hidden_dim, batch_first=True, bidirectional=True)
        self.fc_hidden = nn.Linear(enc_hidden_dim * 2, dec_hidden_dim)
        self.fc_cell = nn.Linear(enc_hidden_dim * 2, dec_hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_lengths):
        embedded = self.dropout(self.embedding(src))
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            src_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_outputs, (hidden, cell) = self.rnn(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True)

        hidden_cat = torch.cat((hidden[-2], hidden[-1]), dim=1)
        cell_cat = torch.cat((cell[-2], cell[-1]), dim=1)

        decoder_hidden = torch.tanh(self.fc_hidden(hidden_cat)).unsqueeze(0)
        decoder_cell = torch.tanh(self.fc_cell(cell_cat)).unsqueeze(0)
        return outputs, decoder_hidden, decoder_cell

class BahdanauAttention(nn.Module):
    def __init__(self, enc_hidden_dim, dec_hidden_dim):
        super().__init__()
        self.attn = nn.Linear((enc_hidden_dim * 2) + dec_hidden_dim, dec_hidden_dim)
        self.v = nn.Linear(dec_hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs, mask):
        src_len = encoder_outputs.shape[1]
        repeated_hidden = hidden[-1].unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((repeated_hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        attention = attention.masked_fill(mask == 0, -1e10)
        return torch.softmax(attention, dim=1)

class Decoder(nn.Module):
    def __init__(self, output_dim, embed_dim, enc_hidden_dim, dec_hidden_dim, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, embed_dim, padding_idx=PAD_IDX)
        self.rnn = nn.LSTM((enc_hidden_dim * 2) + embed_dim, dec_hidden_dim, batch_first=True)
        self.fc_out = nn.Linear((enc_hidden_dim * 2) + dec_hidden_dim + embed_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, cell, encoder_outputs, mask):
        input_token = input_token.unsqueeze(1)
        embedded = self.dropout(self.embedding(input_token))
        attention_weights = self.attention(hidden, encoder_outputs, mask).unsqueeze(1)
        context = torch.bmm(attention_weights, encoder_outputs)
        rnn_input = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))

        prediction = self.fc_out(torch.cat((output.squeeze(1), context.squeeze(1), embedded.squeeze(1)), dim=1))
        return prediction, hidden, cell, attention_weights.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def create_mask(self, src):
        return src != PAD_IDX

    def forward(self, src, src_lengths, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        tgt_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, tgt_len - 1, tgt_vocab_size, device=self.device)
        encoder_outputs, hidden, cell = self.encoder(src, src_lengths)
        mask = self.create_mask(src)

        input_token = tgt[:, 0]
        for t in range(1, tgt_len):
            output, hidden, cell, _ = self.decoder(input_token, hidden, cell, encoder_outputs, mask)
            outputs[:, t - 1, :] = output
            use_teacher_forcing = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_token = tgt[:, t] if use_teacher_forcing else top1

        return outputs

In [9]:
attention = BahdanauAttention(ENC_HIDDEN_DIM, DEC_HIDDEN_DIM)
encoder = Encoder(len(src_itos), EMBED_DIM, ENC_HIDDEN_DIM, DEC_HIDDEN_DIM, DROPOUT)
decoder = Decoder(len(tgt_itos), EMBED_DIM, ENC_HIDDEN_DIM, DEC_HIDDEN_DIM, DROPOUT, attention)
model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable parameters: {count_parameters(model):,}")

Trainable parameters: 11,371,492


## Training

In [10]:
def train_one_epoch(model, loader, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0.0

    for src, src_lengths, tgt in tqdm(loader, desc="train", leave=False):
        src = src.to(DEVICE)
        src_lengths = src_lengths.to(DEVICE)
        tgt = tgt.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        output = model(src, src_lengths, tgt, TEACHER_FORCING_RATIO)
        output_dim = output.shape[-1]

        output = output.reshape(-1, output_dim)
        target = tgt[:, 1:].reshape(-1)
        loss = criterion(output, target)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0.0

    for src, src_lengths, tgt in tqdm(loader, desc="valid", leave=False):
        src = src.to(DEVICE)
        src_lengths = src_lengths.to(DEVICE)
        tgt = tgt.to(DEVICE)

        output = model(src, src_lengths, tgt, teacher_forcing_ratio=0.0)
        output_dim = output.shape[-1]

        output = output.reshape(-1, output_dim)
        target = tgt[:, 1:].reshape(-1)
        loss = criterion(output, target)
        epoch_loss += loss.item()

    return epoch_loss / len(loader)

def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [11]:
best_valid_loss = float("inf")
checkpoint_path = "/kaggle/working/lstm_attention_summarizer.pt" if os.path.exists("/kaggle/working") else "lstm_attention_summarizer.pt"

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, valid_loader, criterion)

    end_time = time.time()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "src_stoi": src_stoi,
                "src_itos": src_itos,
                "tgt_stoi": tgt_stoi,
                "tgt_itos": tgt_itos,
                "config": {
                    "max_source_len": MAX_SOURCE_LEN,
                    "max_target_len": MAX_TARGET_LEN,
                    "embed_dim": EMBED_DIM,
                    "enc_hidden_dim": ENC_HIDDEN_DIM,
                    "dec_hidden_dim": DEC_HIDDEN_DIM,
                    "dropout": DROPOUT,
                },
            },
            checkpoint_path,
        )

    print(f"Epoch: {epoch:02} | Time: {epoch_mins}m {epoch_secs}s")
    print(f"\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(min(train_loss, 20)):.2f}")
    print(f"\tValid Loss: {valid_loss:.3f} | Valid PPL: {math.exp(min(valid_loss, 20)):.2f}")

print("Best checkpoint saved to:", checkpoint_path)

train:   0%|          | 0/1016 [00:00<?, ?it/s]

valid:   0%|          | 0/113 [00:00<?, ?it/s]

Epoch: 01 | Time: 1m 39s
	Train Loss: 5.213 | Train PPL: 183.56
	Valid Loss: 5.121 | Valid PPL: 167.50


train:   0%|          | 0/1016 [00:00<?, ?it/s]

valid:   0%|          | 0/113 [00:00<?, ?it/s]

...

valid:   0%|          | 0/113 [00:00<?, ?it/s]

Epoch: 04 | Time: 1m 40s
	Train Loss: 4.096 | Train PPL: 60.12
	Valid Loss: 5.046 | Valid PPL: 155.41


train:   0%|          | 0/1016 [00:00<?, ?it/s]

valid:   0%|          | 0/113 [00:00<?, ?it/s]

Epoch: 05 | Time: 1m 40s
	Train Loss: 3.884 | Train PPL: 48.63
	Valid Loss: 5.112 | Valid PPL: 166.01
Best checkpoint saved to: /kaggle/working/lstm_attention_summarizer.pt


## Generate Summaries

In [12]:
@torch.no_grad()
def summarize(review, model, max_len=MAX_TARGET_LEN):
    model.eval()
    cleaned = clean_text(review)
    ids = encode_source(cleaned)
    if not ids:
        ids = [UNK_IDX]

    src = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    src_lengths = torch.tensor([len(ids)], dtype=torch.long, device=DEVICE)

    encoder_outputs, hidden, cell = model.encoder(src, src_lengths)
    mask = model.create_mask(src)

    input_token = torch.tensor([SOS_IDX], dtype=torch.long, device=DEVICE)
    generated_tokens = []
    attention_history = []

    for _ in range(max_len):
        output, hidden, cell, attention_weights = model.decoder(input_token, hidden, cell, encoder_outputs, mask)
        top1 = output.argmax(1).item()
        if top1 == EOS_IDX:
            break
        if top1 not in (PAD_IDX, SOS_IDX):
            generated_tokens.append(tgt_itos[top1])
            attention_history.append(attention_weights.squeeze(0).detach().cpu().numpy())
        input_token = torch.tensor([top1], dtype=torch.long, device=DEVICE)

    return " ".join(generated_tokens), attention_history

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])

for i in range(min(5, len(valid_df))):
    review = valid_df.loc[i, "Text"]
    actual = valid_df.loc[i, "Summary"]
    predicted, _ = summarize(review, model)
    print("Review:", review[:450].replace("\n", " "))
    print("Actual summary:", actual)
    print("Predicted summary:", predicted)
    print("-" * 100)

Review: I found this blend to be a bit lacking in depth. Its OK, but not as intense as I'd like. It won't be replacing my standard favorite.
Actual summary: Just OK
Predicted summary: not blend
----------------------------------------------------------------------------------------------------
Review: I've tried so many different teas wanting to take the healthy approach but have been shoved right back to the coffee pot due to complete lack of flavor.  Tea always seemed so boring.  However, this tea is so yummy I am converted. Don't get me wrong, I still enjoy coffee, but I now incorporate this tea almost daily.  I love the Chamomile Citrus and Green tea Tropical!!  The smell is divine and the tea bags are practically a work of art...  LOVES I
Actual summary: Finally!! A great tasting tea :o)
Predicted summary: great tea
----------------------------------------------------------------------------------------------------
Review: I placed an order for 4 separate Kringle's to go to 4 sepa

## Try Your Own Review

In [13]:
my_review = """
I bought this coffee because I wanted something smooth for mornings. The flavor is rich without being too bitter, and the aroma is excellent. The package arrived fresh and sealed. I would definitely buy it again, although the price could be a little lower.
"""

summary, _ = summarize(my_review, model)
print("Generated summary:", summary)

Generated summary: great coffee


## Optional: Simple ROUGE-1 F1 Check

This lightweight metric is included without installing extra packages. It is not a substitute for a full evaluation library, but it gives a quick sanity check.

In [14]:
def rouge1_f1(reference, prediction):
    ref_tokens = clean_text(reference).split()
    pred_tokens = clean_text(prediction).split()
    if not ref_tokens or not pred_tokens:
        return 0.0

    ref_counts = Counter(ref_tokens)
    pred_counts = Counter(pred_tokens)
    overlap = sum(min(ref_counts[token], pred_counts[token]) for token in pred_counts)
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

sample_size = min(100, len(valid_df))
scores = []
for i in tqdm(range(sample_size), desc="rouge1"):
    predicted, _ = summarize(valid_df.loc[i, "Text"], model)
    scores.append(rouge1_f1(valid_df.loc[i, "Summary"], predicted))

print(f"Mean ROUGE-1 F1 on {sample_size} validation examples: {np.mean(scores):.4f}")

rouge1:   0%|          | 0/100 [00:00<?, ?it/s]

Mean ROUGE-1 F1 on 100 validation examples: 0.1919
